# RETAIL DATA PROJECT 

## Reading Bronze Data 

In [16]:
# Load messy raw data into DataFrames from CSVs
df_orders_raw = spark.read.parquet("abfss://ec22e669-2f20-440d-8f2c-736bee24d45f@onelake.dfs.fabric.microsoft.com/505c1ba4-535c-49df-8635-92f052a10e43/Files/bronze/orders_data.parquet")
df_returns_raw = spark.read.parquet("abfss://ec22e669-2f20-440d-8f2c-736bee24d45f@onelake.dfs.fabric.microsoft.com/505c1ba4-535c-49df-8635-92f052a10e43/Files/bronze/returns_data.xlsx.parquet")
df_inventory_raw = spark.read.parquet("abfss://ec22e669-2f20-440d-8f2c-736bee24d45f@onelake.dfs.fabric.microsoft.com/505c1ba4-535c-49df-8635-92f052a10e43/Files/bronze/inventory_data.parquet")

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 20, Finished, Available, Finished, False)

In [5]:
display(df_inventory_raw) 

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4872ca86-20ff-43cf-bbe8-ad0afbb07b2f)

In [11]:
display(df_orders_raw)

StatementMeta(, a5be52f2-56cc-46c3-b10f-95f8a7192a39, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 275da45c-63c8-4bd8-aded-208f5cff28d2)

In [17]:
display(df_returns_raw)

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 611c323a-b40c-4fa1-8d25-5dc36447dd18)

In [18]:
# Extract first row as header
first_row = df_returns_raw.first()
columns = [str(item).strip() for item in first_row]

# Remove the first row (header row now part of data)
df_returns = df_returns_raw.rdd.zipWithIndex().filter(lambda x: x[1] > 0).map(lambda x: x[0]).toDF(columns)

# Show cleaned data
display(df_returns)

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c219c5d9-3aa3-41cf-95d1-2b665cd2ff3d)

In [19]:
df_returns.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_returns")

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 23, Finished, Available, Finished, False)

## Creating bronze delta tables

In [8]:
df_orders_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_orders")
df_returns_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_returns")
df_inventory_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_inventory")

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 12, Finished, Available, Finished, False)

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 5, Finished, Available, Finished, False)

NameError: name 'df_returns' is not defined

# Cleaning data - Silver Layer

In [21]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 25, Finished, Available, Finished, False)

#### Cleaning Order Table

In [18]:
from pyspark.sql.functions import *

df_orders = (
    df_orders_raw
    # Fix inconsistent column names
    .withColumnRenamed("Order iD", "Order_ID")
    .withColumnRenamed("order_date", "Order_Date")

    # Convert Order_Date to proper format, replacing "/" with "-" and casting to date
    .withColumn("Order_Date", to_date(regexp_replace(col("Order_Date"), "/", "-"), "dd-MM-yyyy"))

    # Standardize Customer ID format – uppercase and trimmed
    .withColumn("cust_id", trim(upper(col("cust_id"))))

    # Clean emails – lowercase and trimmed
    .withColumn("Email", lower(trim(col("Email"))))

    # Handle empty/null payment modes
    .withColumn("Payment_Mode", when(length(trim(col("Payment_Mode"))) == 0, "unknown").otherwise(col("Payment_Mode")))

    # Replace null promo codes with placeholder
    .withColumn("Promo_Code", coalesce(col("Promo_Code"), lit("NO_PROMO")))

    # Fix negative order amounts and cast to double
    .withColumn("Order_Amount", abs(col("Order_Amount$").cast("double")))

    # Add year and month columns for easier reporting
    .withColumn("Order_Year", year("Order_Date"))
    .withColumn("Order_Month", month("Order_Date"))

    # Create unique hash key for deduplication or tracking
    .withColumn("Order_Hash", sha2(concat_ws("|", *df_orders_raw.columns), 256))

    # Drop rows where critical fields are missing
    .dropna(subset=["Order_ID", "Order_Amount", "cust_id"])

    # Remove duplicate Order IDs
    .dropDuplicates(["Order_ID"])
)

StatementMeta(, a5be52f2-56cc-46c3-b10f-95f8a7192a39, 27, Finished, Available, Finished, False)

In [19]:
df_orders.write.mode("overwrite").format("delta").saveAsTable("silver_orders")

StatementMeta(, a5be52f2-56cc-46c3-b10f-95f8a7192a39, 28, Finished, Available, Finished, False)

#### Cleaning Inventory Data

In [24]:
df_inventory = (
    df_inventory_raw
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("last_stocked", "LastStocked")
    
    # 2. Clean stock column: convert to integer
    .withColumn("Stock", 
        when(col("stock").rlike("^[0-9]+$"), col("stock").cast(IntegerType()))  # Numeric values
        .when(col("stock").isNull() | (col("stock") == ""), lit(None))  # Null or blank
        .otherwise(
            when(col("stock").rlike(".*twenty five.*"), lit(25))
            .when(col("stock").rlike(".*twenty.*"), lit(20))
            .when(col("stock").rlike(".*eighteen.*"), lit(18))
            .when(col("stock").rlike(".*fifteen.*"), lit(15))
            .when(col("stock").rlike(".*twelve.*"), lit(12))
            .otherwise(lit(None))
        ).cast(IntegerType())
    )
    
    # 3. Clean LastStocked: normalize multiple date formats to yyyy-MM-dd
    .withColumn("LastStocked", to_date(
        regexp_replace("LastStocked", "[./]", "-"), "yyyy-MM-dd"
    ))
    
    # 4. Clean CostPrice: extract numeric value and convert to float
    .withColumn("CostPrice", 
        regexp_extract(col("CostPrice"), r"(\d+\.?\d*)", 1).cast(DoubleType())
    )

    # 5. Clean Warehouse: remove special characters, trim, capitalize first letter
    .withColumn("Warehouse", 
        initcap(trim(regexp_replace(col("warehouse"), r"[^a-zA-Z0-9\s]", " ")))
    )
    
    # 6. Standardize Available: convert to boolean
    .withColumn("Available", 
        when(lower(col("available")).isin("yes", "y", "true"), lit(True))
        .when(lower(col("available")).isin("no", "n", "false"), lit(False))
        .otherwise(None)
    )
    
    # 7. Drop raw messy columns
    .drop("stock", "warehouse", "available")
)

StatementMeta(, a5be52f2-56cc-46c3-b10f-95f8a7192a39, 33, Finished, Available, Finished, False)

In [3]:
df_inventory.show()

df_inventory.write.mode("overwrite").format("delta").saveAsTable("silver_inventory")


StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 6, Finished, Available, Finished, False)

NameError: name 'df_inventory' is not defined

#### Cleaning Returns Data  

In [24]:
df_returns = (

   df_returns
    # 2.1 Standardize column names (if needed)
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")
    
    # 2.2 Clean ReturnDate → standardize date formats
    .withColumn("ReturnDate", to_date(
        regexp_replace("ReturnDate", r"[./]", "-"), "dd-MM-yyyy"
    ))
    
    # 2.3 Clean RefundStatus → lowercase, remove special characters
    .withColumn("RefundStatus", lower(regexp_replace(col("RefundStatus"), r"[^a-zA-Z]", "")))
    
    # 2.4 Clean ReturnAmount → extract numeric part regardless of currency
    .withColumn("ReturnAmount", 
        regexp_extract(col("ReturnAmount"), r"(\d+\.?\d*)", 1).cast(DoubleType())
    )
    
    # 2.5 Clean PickupAddress → remove special characters
    .withColumn("PickupAddress", initcap(trim(regexp_replace(col("PickupAddress"), r"[^a-zA-Z0-9\s]", " "))))
    
    # 2.6 Clean Product → remove extra symbols and spaces
    .withColumn("Product", initcap(trim(regexp_replace(col("Product"), r"[^a-zA-Z0-9\s]", ""))))
    
    # 2.7 Clean CustomerID → trim, fix wrong prefixes
    .withColumn("CustomerID", trim(upper(col("CustomerID"))))
    
    # 2.8 Drop rows with null ReturnID (R014)
    .filter(col("ReturnID").isNotNull())
)

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 28, Finished, Available, Finished, False)

In [28]:
df_returns = df_returns.withColumn("ReturnReason", lower(col("ReturnReason")))

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 32, Finished, Available, Finished, False)

In [29]:
display(df_returns)

# Step 4: Save to Silver Delta Table
df_returns.write.mode("overwrite").format("delta").saveAsTable("silver_returns")

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ba279181-b8d4-44e9-8190-1e7de1544079)

In [51]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_orders_raw = spark.table("bronze_orders")

df_orders = (
    df_orders_raw
    # Extract numeric portion from Order_Amount$ (strips $, ₹, "USD", etc.)
    .withColumn("Order_Amount", abs(regexp_extract(col("Order_Amount$"), r"(\d+\.?\d*)", 1).cast(DoubleType())))

    # Normalize multiple date formats — try each pattern in order
    .withColumn("Order_Date_clean", regexp_replace(col("Order_Date"), r"[./]", "-"))
    .withColumn("Order_Date",
        coalesce(
            to_date(col("Order_Date_clean"), "yyyy-MM-dd"),
            to_date(col("Order_Date_clean"), "dd-MM-yyyy"),
            to_date(col("Order_Date_clean"), "MM-dd-yyyy")
        )
    )
    .drop("Order_Date_clean")

    # Standardize cust_id (handles "C_002" -> "C002")
    .withColumn("cust_id", trim(upper(regexp_replace(col("cust_id"), "_", ""))))

    # Clean product name
    .withColumn("Product_Name", initcap(trim(col("Product_Name"))))

    # Clean email
    .withColumn("Email", lower(trim(col("Email"))))

    # Handle empty/null payment modes
    .withColumn("Payment_Mode", when(trim(coalesce(col("Payment_Mode"), lit(""))) == "", "unknown").otherwise(col("Payment_Mode")))

    # Replace null promo codes
    .withColumn("Promo_Code", coalesce(col("Promo_Code"), lit("NO_PROMO")))

    # Add year/month
    .withColumn("Order_Year", year("Order_Date"))
    .withColumn("Order_Month", month("Order_Date"))

    # Hash
    .withColumn("Order_Hash", sha2(concat_ws("|", *df_orders_raw.columns), 256))

    # Drop rows missing only truly critical fields — Order_ID and Order_Amount
    .dropna(subset=["Order_ID", "Order_Amount"])

    .dropDuplicates(["Order_ID"])
)

display(df_orders)

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f627b86-3e92-4e00-bcd3-526feb9a5998)

## GOLD LAYER – Aggregation & KPIs
#### Enrich Orders with Returns & Inventory


In [56]:
from pyspark.sql.functions import *

# STEP 1: Load cleaned Silver tables with aliases
orders = spark.table("silver_orders").alias("o")
returns = spark.table("silver_returns").alias("r")
inventory = spark.table("silver_inventory").alias("i")

# STEP 2: Join Orders with Returns (LEFT)
order_return = orders.join(
    returns,
    on=col("o.Order_ID") == col("r.OrderID"),
    how="left"
)

# STEP 3: Join with Inventory (LEFT)
enriched = order_return.join(
    inventory,
    on=col("o.Product_Name") == col("i.ProductName"),
    how="left"
)

# STEP 4: Add derived column for month (e.g., "2023-07")
enriched = enriched.withColumn("OrderMonth", date_format(col("o.Order_Date"), "yyyy-MM"))

# STEP 5: Select explicit columns to avoid ambiguity
df_enriched = enriched.select(
    col("o.Product_Name").alias("ProductName"),
    col("o.Order_ID").alias("OrderID"),
    col("o.cust_id").alias("CustomerID"),
    col("o.Order_Amount").alias("OrderAmount"),
    col("r.ReturnID").alias("ReturnID"),
    col("i.CostPrice").alias("CostPrice"),
    col("OrderMonth")
)

# STEP 6: Aggregate KPIs by Product and Month
df_kpi = (
    df_enriched.groupBy("ProductName", "OrderMonth")
    .agg(
        count("OrderID").alias("Total_Orders"),
        countDistinct("CustomerID").alias("Unique_Customers"),
        count("ReturnID").alias("Total_Returns"),
        round((count("ReturnID") / count("OrderID")) * 100, 2).alias("Return_Rate_Pct"),
        round(sum("OrderAmount"), 2).alias("Total_Revenue"),
        round(avg("OrderAmount"), 2).alias("Avg_Order_Value"),
        round(avg("CostPrice"), 2).alias("Avg_Cost")
    )
)

# STEP 7: Display results
display(df_kpi)

# STEP 8: Save to Gold Delta table
# df_kpi.write.mode("overwrite").format("delta").saveAsTable("gold_product_month_kpis")

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72457a31-40c3-454c-be65-85cfe5e58650)

In [58]:

# STEP 8: Save to Gold Delta table
df_kpi.write.mode("overwrite").format("delta").saveAsTable("gold_product_month_kpis")

StatementMeta(, 6f84f42c-9b6b-4fe5-a828-25a1ec374f6f, 62, Finished, Available, Finished, False)